# Shadow Testing: Multi-Model Migration Evaluation (CloudWatch Logs)

Shadow testing lets you evaluate candidate models using **real production traffic** before committing to a migration. Instead of synthetic benchmarks, you compare models on the actual queries your system handles every day.

This notebook extracts **real invocation logs from Amazon Bedrock's CloudWatch logging** and uses them to evaluate candidate models:

1. **Extract invocation logs** — Pull real production traffic from CloudWatch Logs
2. **Sample from logs** — Randomly select representative samples for evaluation
3. **Evaluate target models** — Run sampled prompts against candidate models
4. **Rank on a leaderboard** — Compare accuracy (LLM-as-a-Judge), latency, and cost

## Architecture

![Shadow Testing Architecture](data/shadow-test-architecture.png)

---

## Step 1: Setup

We import helper functions from `src/utils.py` for model evaluation and from `src/invocation_logs.py` for extracting real Bedrock invocation logs from CloudWatch.

In [ ]:
import json
import time
import random
from typing import Dict, List
from dataclasses import dataclass, field
import numpy as np
import pandas as pd
import plotly.graph_objects as go

import sys
sys.path.insert(0, '.')

from src.utils import (
    create_bedrock_client,
    compute_model_cost,
    run_model_evaluation,
    compare_models,
    EvaluationResult,
    QUALITY_METRICS,
    print_evaluation_progress,
    print_evaluation_summary,
    print_migration_recommendation,
    build_leaderboard,
    render_leaderboard_table,
    render_quality_breakdown_table,
)

from src.invocation_logs import (
    print_logging_status,
    extract_logs_from_cloudwatch,
    sample_logs,
)

REGION = "us-east-1"
bedrock = create_bedrock_client(REGION)

# Production model — the model we're extracting logs for
PRODUCTION_MODEL_ID = "us.amazon.nova-2-lite-v1:0"
PRODUCTION_MODEL_NAME = "Amazon Nova 2 Lite"

# Target models to evaluate as potential replacements
TARGET_MODELS = {
    "qwen.qwen3-coder-30b-a3b-v1:0": "Qwen 3 Coder",
    "us.amazon.nova-2-lite-v1:0": "Amazon Nova 2 Lite",
    "openai.gpt-oss-20b-1:0": "OpenAI GPT-OSS 20B",
    "us.anthropic.claude-haiku-4-5-20251001-v1:0": "Claude Haiku 4.5",
}

# Judge models for LLM-as-a-Judge evaluation
JUDGE_MODEL_IDS = [
    "us.anthropic.claude-3-5-haiku-20241022-v1:0",
    "us.amazon.nova-pro-v1:0",
    "moonshotai.kimi-k2.5",
]

# How many samples to evaluate
SAMPLE_SIZE = 10

print(f"Production model:  {PRODUCTION_MODEL_NAME}")
print(f"Target models:     {len(TARGET_MODELS)}")
for mid, name in TARGET_MODELS.items():
    print(f"  - {name}")
print(f"Judge models:      {len(JUDGE_MODEL_IDS)}")
print(f"Sample size:       {SAMPLE_SIZE}")

---

## Step 2: Extract Invocation Logs from CloudWatch

Amazon Bedrock's **invocation logging** captures every model request and response when enabled. Logs are written to CloudWatch Logs and include the full request body, response body, token counts, and latency.

We extract logs for the production model directly from CloudWatch — no simulation needed. These are real queries that users sent to the production system.

In [ ]:
# Check that invocation logging is enabled
print_logging_status(REGION)

### Invocation Log Schema

Each CloudWatch log entry is a JSON record captured by Bedrock. The fields we use:

| Field | Path in log | Description |
|-------|------------|-------------|
| Request ID | `requestId` | Unique identifier for the invocation |
| Timestamp | `timestamp` | UTC time of the request |
| Model ID | `modelId` | Which Bedrock model was called |
| Operation | `operation` | API used (`Converse`, `InvokeModel`) |
| Input tokens | `input.inputTokenCount` | Tokens in the request |
| Output tokens | `output.outputTokenCount` | Tokens in the response |
| Latency | `output.outputBodyJson.metrics.latencyMs` | Server-side latency measured by Bedrock |
| User prompt | `input.inputBodyJson.messages[].content[].text` | The actual prompt sent by the user |
| Model response | `output.outputBodyJson.output.message.content[].text` | The model's response text |

In [ ]:
# Extract production model invocation logs from CloudWatch
all_logs = extract_logs_from_cloudwatch(
    model_id_filter="nova-2-lite",
    max_records=200,
    region=REGION,
)

# Filter out judge/evaluator prompts (keep only real user traffic)
invocation_logs = [l for l in all_logs if "expert evaluator" not in l.prompt[:50] and l.prompt]

print(f"\nProduction logs ({PRODUCTION_MODEL_NAME}): {len(invocation_logs)}")
print(f"\nSample prompts:")
for log in invocation_logs[:5]:
    print(f"  {log.latency_ms:6.0f}ms | {log.input_tokens:3d}in/{log.output_tokens:3d}out | {log.prompt[:70]}...")

In [ ]:
# Invocation log summary
total_tokens = sum(l.output_tokens for l in invocation_logs)
total_latency = np.mean([l.latency_ms for l in invocation_logs])

print("INVOCATION LOG SUMMARY (from CloudWatch)")
print("=" * 60)
print(f"  Source:              CloudWatch Logs")
print(f"  Production model:    {PRODUCTION_MODEL_NAME}")
print(f"  Total logs:          {len(invocation_logs)}")
print(f"  Total output tokens: {total_tokens:,}")
print(f"  Avg latency:         {total_latency:.0f} ms")

---

## Step 3: Sample from Invocation Logs

We randomly select entries from the extracted CloudWatch logs. These are real production queries that we'll replay against all target models.

In [ ]:
# Randomly sample from invocation logs
sampled_logs = sample_logs(invocation_logs, n=SAMPLE_SIZE)

sample_rows = []
for log in sampled_logs:
    sample_rows.append({
        "Request ID": log.request_id[:12] + "...",
        "Timestamp": log.timestamp,
        "Prompt": log.prompt[:70] + "...",
        "Prod Latency": f"{log.latency_ms:.0f}ms",
        "Prod Tokens": log.output_tokens,
    })

df_samples = pd.DataFrame(sample_rows)
print(df_samples.to_string(index=False))

---

## Step 4: Evaluate Target Models

For each sampled prompt, we invoke all target models and evaluate their responses using:

- **LLM-as-a-Judge** — 3 judge models score each response on 6 quality dimensions. Majority vote determines PASS/FAIL.
- **Golden answer** — We use the production model's actual response as the reference answer, since these are real queries without curated golden answers.
- **Latency and cost** — Captured directly from each invocation.

In [ ]:
# Evaluate all target models on the sampled prompts (parallelized per model)
import concurrent.futures

evaluation_results: Dict[str, List[EvaluationResult]] = {}

for model_id, model_name in TARGET_MODELS.items():
    print(f"\nEvaluating: {model_name}")

    # Run all sampled prompts in parallel for this model
    with concurrent.futures.ThreadPoolExecutor(max_workers=len(sampled_logs)) as executor:
        future_to_log = {
            executor.submit(
                run_model_evaluation,
                prompt_id=log.request_id,
                user_prompt=log.prompt,
                model_id=model_id,
                judge_model_ids=JUDGE_MODEL_IDS,
                golden_answer=log.response,  # Use production response as golden answer
                bedrock_client=bedrock,
                region=REGION,
            ): log
            for log in sampled_logs
        }

        model_results = []
        for future in concurrent.futures.as_completed(future_to_log):
            log = future_to_log[future]
            result = future.result()
            model_results.append((log, result))
            print_evaluation_progress(result, log.request_id[:12], log.prompt[:40])

    # Preserve original sample order
    log_order = {log.request_id: i for i, log in enumerate(sampled_logs)}
    model_results.sort(key=lambda x: log_order[x[0].request_id])
    evaluation_results[model_id] = [r for _, r in model_results]

print_evaluation_summary(evaluation_results, TARGET_MODELS, len(sampled_logs))

---

## Step 5: Leaderboard

We aggregate the evaluation results into a **leaderboard** that ranks each target model on three axes:

| Axis | What it measures | How |
|------|-----------------|-----|
| **Accuracy** | Response quality and medical safety | LLM-as-a-Judge pass rate + avg quality score + safety score |
| **Latency** | Response speed | Average time-to-last-byte (ms) |
| **Cost** | Per-query expense | Average cost per invocation ($) |

In [ ]:
# Build the leaderboard
prod_avg_latency = np.mean([l.latency_ms for l in sampled_logs])
prod_avg_cost = np.mean([
    compute_model_cost(l.input_tokens, l.output_tokens, PRODUCTION_MODEL_ID)
    for l in sampled_logs
])

leaderboard, comparison = build_leaderboard(
    evaluation_results, TARGET_MODELS, prod_avg_latency, prod_avg_cost
)

render_leaderboard_table(leaderboard)

In [ ]:
render_quality_breakdown_table(leaderboard, comparison)

---

## Step 6: Migration Recommendation

Based on the leaderboard, we identify the best candidate for migration — the model with the strongest combination of accuracy, safety, cost, and latency for this healthcare Q&A workload.

In [ ]:
print_migration_recommendation(
    best=leaderboard[0],
    prod_model_name=PRODUCTION_MODEL_NAME,
    prod_avg_latency=prod_avg_latency,
    prod_avg_cost=prod_avg_cost,
    quality_rec_model_id=comparison.recommended_model,
    model_names=TARGET_MODELS,
)

---

## Traffic Migration Strategy

Once you've identified your target model, migrate production traffic gradually using canary deployment:

| Phase | Traffic to Target | Duration | Success Criteria |
|-------|-------------------|----------|------------------|
| Canary | 5% | 1-2 days | No increase in errors, latency within bounds |
| Early Adopters | 25% | 3-5 days | Quality maintained, user feedback positive |
| Majority | 75% | 1 week | All metrics stable, cost projections confirmed |
| Full Migration | 100% | Ongoing | Source model decommissioned |

**Rollback triggers** — automatically revert if:
- Error rate increases by more than 50%
- P95 latency exceeds 2x baseline for 5+ minutes
- Safety score drops below 0.60 (critical medical entities missing)

In production, implement the shadow router using feature flags (AWS AppConfig), weighted target groups (ALB), or a custom routing layer in your API gateway.

---

## Summary

This notebook demonstrated a shadow testing workflow using **real production logs**:

1. **Extracted invocation logs from CloudWatch** — real Bedrock traffic, not simulated
2. **Sampled representative queries** from the logs for cost-effective evaluation
3. **Evaluated 4 target models** using LLM-as-a-Judge (3 judges, 6 quality metrics)
4. **Ranked models on a leaderboard** by accuracy, latency, and cost (weighted composite score)

The key difference from the simulated version: **these are real production queries** captured by Bedrock's invocation logging. The production model's actual responses serve as the golden answer for judge evaluation.

| Notebook | What You Learned |
|----------|------------------|
| 1. Drift Detection | How model outputs change over time and across prompt variations |
| 2. Model Evaluation | How to systematically score models using LLM-as-a-Jury |
| 3. Prompt Optimization | How to write prompts that work consistently across models |
| 4. Shadow Testing | How to use production traffic to validate a migration decision |
| **4b. Shadow Testing (CloudWatch)** | **How to extract real Bedrock logs and evaluate candidates against production** |